In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("Project root added:", PROJECT_ROOT)

Project root added: c:\Users\Bluepal\Desktop\AI_RESEARCH_AGENT


# AI Research Agent — Exploration Notebook

**Purpose:**  
This notebook demonstrates the internal behavior of the AI Research Agent system.  
It runs the full pipeline once and inspects:

- Research planning
- Iterative discovery
- Vector memory reuse vs web ingestion
- Agreement reasoning
- Summary scoring
- Final evaluation

This notebook is for **system understanding**, not benchmarking.

In [7]:
from src.controller.run import run_pipeline
from src.vector_store.client import VectorStoreClient
from src.trace.research_trace import ResearchTrace

import pandas as pd
import textwrap

c:\Users\Bluepal\Desktop\AI_RESEARCH_AGENT\venv\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.0) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
c:\Users\Bluepal\Desktop\AI_RESEARCH_AGENT\src\analytics\agreement_detector.py:4: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


## 1️⃣ Define Query & Mode

In [8]:
query = "Assess the influence of Virat Kohli on fitness culture and practices in India"
mode = "standard"   # quick | standard | deep

## 2️⃣ Run Full Research Pipeline

In [9]:
vector_client = VectorStoreClient(embedding_dim=384)
trace = ResearchTrace()

summaries, trace_text, report_text, pdf_path, evaluation = run_pipeline(
    user_query=query,
    mode=mode,
    vector_client=vector_client,
    trace=trace,
)

print("Pipeline complete.")

Pipeline complete.


## 3️⃣ Research Plan

In [10]:
plan_section = trace_text.split("DISCOVERY ITERATION")[0]
print(plan_section)


RESEARCH PLAN
Goal:
Evaluate the impact of Virat Kohli on fitness trends and behaviors in India

Dimensions:
- Cultural Influence and Public Perception
- Behavioral Changes in Fitness Practices
- Media and Social Media Amplification
- Industry Response and Market Trends
- Long-term Sustainability of Fitness Culture
- Comparative Analysis with Other Influencers




## 4️⃣ Discovery Iterations Overview

In [11]:
for line in trace_text.splitlines():
    if "DISCOVERY ITERATION" in line:
        print(line)

DISCOVERY ITERATION 1
DISCOVERY ITERATION 2


## 5️⃣ Vector Reuse vs Web Ingestion
Shows when the system reused memory vs searched the web.

In [12]:
for line in trace_text.splitlines():
    if "Reused summary" in line or "WEB INGESTION" in line:
        print(line)

WEB INGESTION — Q1
WEB INGESTION — Q2
WEB INGESTION — Q1
WEB INGESTION — Q2


## 6️⃣ Agreement Map (Cross-Source Reasoning)

In [13]:
start = trace_text.find("AGREEMENT MAP")
end = trace_text.find("TOTAL SCORES")
print(trace_text[start:end])

AGREEMENT MAP
S1 → S2: partially_supports
S1 → S3: independent
S1 → S4: partially_supports
S2 → S1: partially_supports
S2 → S3: independent
S2 → S4: partially_supports
S3 → S1: independent
S3 → S2: independent
S3 → S4: independent
S4 → S1: partially_supports
S4 → S2: partially_supports
S4 → S3: independent




## 7️⃣ Summary Scoring Table

In [14]:
df = pd.DataFrame([
    {
        "ID": s["id"],
        "Credibility": s.get("credibility_score", 0),
        "Agreement": s.get("agreement_score", 0),
        "Total Score": s.get("total_score", 0),
        "Domain": s.get("domain"),
        "URL": s.get("url"),
    }
    for s in summaries
])

df

,ID,Credibility,Agreement,Total Score,Domain,URL
0,S1,0,7,7,dreamindia.substack.com,https://dreamindia.substack.com/p/20-indias-fi...
1,S2,2,7,9,bollywoodshaadis.com,https://www.bollywoodshaadis.com/articles/vira...
2,S3,2,3,5,theconversation.com,https://theconversation.com/ferocity-fitness-a...
3,S4,0,7,7,timesofindia.indiatimes.com,https://timesofindia.indiatimes.com/life-style...


## 8️⃣ Final Evidence Summaries

In [15]:
for s in summaries:
    print(f"\n{s['id']} (Score={s.get('total_score')}):")
    print(textwrap.fill(s["summary"], width=100))


S1 (Score=7):
India's fitness market is forecasted to double from $1.9 billion in 2024 to a $4.5 billion by 2030,
with a 15% annual growth rate, driven by 1.4 billion people reclaiming an ancient wellness heritage
while embracing modern fitness culture. This growth is part of a broader trend where exercise is
becoming increasingly important in Indian mainstream culture, with a shift from viewing fitness as a
distraction to a priority for physical, mental, and social wellbeing. India's vision of good health,
as stated by Prime Minister Narendra Modi, implies ensuring wellness and welfare for everyone, not
just being free of disease. The country's fitness renaissance is characterized by a mix of
traditional practices such as yoga, meditation, and Ayurveda, and modern fitness trends like HIIT,
Pilates, and CrossFit. The yoga market reached $6.37 billion in 2024, projected to grow at 12.4%
annually to $20.5 billion by 2034, making it a significantly larger opportunity than the entire
fitn

## 9️⃣ Generated Report Preview (First 1200 Characters)

In [16]:
print(report_text[:1200])

@@TITLE@@
Assessing the Influence of Virat Kohli on Fitness Culture and Practices in India
@@TITLE@@

@@Executive Summary@@
India's fitness market is forecasted to double from $1.9 billion in 2024 to a $4.5 billion by 2030, with a 15% annual growth rate, driven by 1.4 billion people reclaiming an ancient wellness heritage while embracing modern fitness culture [S1]. This growth is part of a broader trend where exercise is becoming increasingly important in Indian mainstream culture, with a shift from viewing fitness as a distraction to a priority for physical, mental, and social wellbeing [S1]. Virat Kohli, a global icon and a brand that represents cricket to the world, has introduced a new side to cricketers, emphasizing the importance of maintaining a set standard of fitness [S2]. His influence has revolutionized the way cricketers maintain themselves physically, with fitness becoming non-negotiable among Indian cricketers and international teams [S2]. Kohli's transformation from a p

## 🔟 Evaluation Output

In [17]:
evaluation

{'overall_score': 7.5,
 'accuracy': {'score': 8,
  'notes': 'Most claims are supported by the provided summaries, but some sentences lack citations.'},
 'completeness': {'score': 6,
  'notes': 'Some planned dimensions, such as Media and Social Media Amplification, Industry Response and Market Trends, and Comparative Analysis with Other Influencers, are missing or weakly covered.'},
 'citation_quality': {'score': 8,
  'notes': 'Most declarative sentences are properly cited, but some citations are missing or incorrect.'},
 'structure': {'score': 7,
  'notes': 'The report has a logical flow, but some sections are unevenly developed and lack depth.'},
 'limitations': ['Lack of coverage of some planned dimensions',
  'Some sentences lack citations'],
 'confidence_level': 'medium'}

## 1️⃣1️⃣ PDF Path

In [18]:
pdf_path

'report_1769613616.pdf'